# Lab 05 — Wumpus World

The agent must collect the gold and leave the cave. It cannot see the map or its own coordinates: it receives only a set of percepts. It remembers its moves, infers which squares it can visit, and uses **condition–action rules** to decide what to do.

The policy has two parts: hand-written rules for straightforward situations, and episodic learning to decide whether to **take a risk or retreat** when no safe exploration is left. This is a hybrid LCS: it learns the value of a few rules without generating new ones.

## 1. Cave and interface

We use the message names from the *Three kinds of messages* slide. In the simulator, `climb` works only at the starting square, `shoot` fires straight in the direction the agent is facing, and the stench disappears after `scream`. The `bump` and `scream` percepts last for one step. Costs add up: for example, `shoot` with an available arrow costs `−1 − 10`. For the experiments, we randomly place three pits, the gold, and the Wumpus in distinct squares other than the starting square.

In [1]:
from collections import Counter, deque
from dataclasses import dataclass
from random import Random

START = (1, 1)
CELLS = tuple((x, y) for x in range(1, 5) for y in range(1, 5))
BOARD = set(CELLS)
DIRECTIONS = ((1, 0), (0, 1), (-1, 0), (0, -1))  # east, north, west, south
EFFECTORS = {"fwd", "turn_l", "turn_r", "grab", "shoot", "climb"}


def advance(cell, heading):
    x, y = cell
    dx, dy = DIRECTIONS[heading]
    return x + dx, y + dy


def neighbors(cell):
    candidates = [advance(cell, h) for h in range(4)]
    return [n for n in candidates if n in BOARD]


class Cave:
    def __init__(self, gold, wumpus, pits):
        self.gold, self.wumpus, self.pits = gold, wumpus, set(pits)
        self.pos, self.heading = START, 0
        self.has_gold, self.arrow = False, True
        self.outcome = None

    def perceive(self, event=None):
        adjacent = set(neighbors(self.pos))
        signals = set()
        if adjacent & self.pits:
            signals.add("breeze")
        if self.wumpus in adjacent | {self.pos}:
            signals.add("stench")
        if self.pos == self.gold:
            signals.add("glitter")
        if self.pos == START:
            signals.add("at_start")
        if event:
            signals.add(event)
        return signals

    def step(self, action):
        if self.outcome is not None or action not in EFFECTORS:
            raise ValueError("Invalid action or episode already ended")

        reward, event = -1, None
        if action == "turn_l":
            self.heading = (self.heading + 1) % 4
        elif action == "turn_r":
            self.heading = (self.heading - 1) % 4
        elif action == "fwd":
            dest = advance(self.pos, self.heading)
            if dest in BOARD:
                self.pos = dest
                if dest in self.pits or dest == self.wumpus:
                    self.outcome = "death"
                    reward -= 1000
            else:
                event = "bump"
        elif action == "grab" and self.pos == self.gold:
            self.has_gold, self.gold = True, None
        elif action == "shoot" and self.arrow:
            self.arrow = False
            reward -= 10
            target = advance(self.pos, self.heading)
            while target in BOARD:
                if target == self.wumpus:
                    self.wumpus, event = None, "scream"
                    break
                target = advance(target, self.heading)
        elif action == "climb" and self.pos == START:
            self.outcome = "gold" if self.has_gold else "retreat"
            reward += 1000 if self.has_gold else 0

        return self.perceive(event), reward, self.outcome


EXAMPLE = ((2, 3), (1, 3), ((3, 1), (3, 3), (4, 4)))


def random_layout(rng):
    gold, wumpus, *pits = rng.sample([c for c in CELLS if c != START], 5)
    return gold, wumpus, tuple(sorted(pits))

## 2. Memory and inference

The agent knows its starting position. From there, it **tracks its position** using turns, `fwd`, and `bump`. It never reads `cave.pos`. It records breeze and stench for each visited square:

- No breeze means no neighboring square contains a pit. With a breeze, if only one candidate remains, it contains a pit.
- No stench means no neighboring square contains the Wumpus. With a stench, the live Wumpus must be in a neighboring square because the agent already entered the current square safely.
- A square is safe if it is excluded from **both** sets of possible hazards.

In [2]:
class Memory:
    def __init__(self):
        self.pos, self.heading = START, 0
        self.seen = {}
        self.flags = set()
        self.arrow, self.wumpus_dead = True, False

    def observe(self, signals, previous=None):
        if previous == "turn_l":
            self.heading = (self.heading + 1) % 4
        elif previous == "turn_r":
            self.heading = (self.heading - 1) % 4
        elif previous == "fwd" and "bump" not in signals:
            self.pos = advance(self.pos, self.heading)
        elif previous == "grab":  # The grab rule requires glitter.
            self.flags.add("has_gold")
        elif previous == "shoot":
            self.arrow = False

        if "scream" in signals:
            self.wumpus_dead = True
        self.seen[self.pos] = signals & {"breeze", "stench"}

    def hazards(self):
        pits = BOARD - self.seen.keys()
        wumpus = set(pits)
        breeze_clues = []

        for cell, signals in self.seen.items():
            adjacent = set(neighbors(cell))
            if "breeze" in signals:
                breeze_clues.append(adjacent)
            else:
                pits -= adjacent
            if "stench" in signals:
                wumpus &= adjacent
            else:
                wumpus -= adjacent

        if self.wumpus_dead:
            wumpus.clear()
        confirmed_pits = set()
        for adjacent in breeze_clues:
            candidates = adjacent & pits
            if len(candidates) == 1:
                confirmed_pits.update(candidates)
        return pits, wumpus, confirmed_pits

## 3. Movement options

Breadth-first search finds the path with the fewest moves on the **inferred map**, not the simulator's map. The controller turns the first move into a turn or `fwd`; it does not minimize the number of turns. It uses only visited squares to return to the start. When targeting an uncertain square, it crosses only safe squares until the final step. It never deliberately enters a confirmed pit or the known location of the live Wumpus.

In [3]:
def route(start, goals, allowed):
    queue = deque([(start, [])])
    visited = {start}
    while queue:
        cell, path = queue.popleft()
        if cell in goals:
            return path
        for dest in neighbors(cell):
            if dest in allowed and dest not in visited:
                visited.add(dest)
                queue.append((dest, path + [dest]))
    return None


def toward(cell, heading, target, aligned="fwd"):
    x, y = cell
    tx, ty = target
    dx, dy = (tx > x) - (tx < x), (ty > y) - (ty < y)
    turns = (DIRECTIONS.index((dx, dy)) - heading) % 4
    if turns == 0:
        return aligned
    return "turn_r" if turns == 3 else "turn_l"


def proposals(memory):
    pits, wumpus, confirmed_pits = memory.hazards()
    safe = BOARD - pits - wumpus
    pos, heading = memory.pos, memory.heading
    home = route(pos, {START}, memory.seen)
    options = {"home": toward(pos, heading, home[0]) if home else "climb"}

    explore = route(pos, safe - memory.seen.keys(), safe)
    if explore:
        options["explore"] = toward(pos, heading, explore[0])
        return options  # Safe exploration takes priority.

    if memory.arrow and len(wumpus) == 1:
        (target,) = wumpus
        firing_cells = {c for c in safe if c[0] == target[0] or c[1] == target[1]}
        aim = route(pos, firing_cells, safe)
        if aim is not None:
            options["hunt"] = toward(
                pos,
                heading,
                aim[0] if aim else target,
                aligned="fwd" if aim else "shoot",
            )
            return options

    uncertain = BOARD - safe - confirmed_pits
    if len(wumpus) == 1:
        uncertain -= wumpus
    risk = route(pos, uncertain, safe | uncertain)
    if risk:
        options["risk"] = toward(pos, heading, risk[0])
    return options

## 4. Classifiers and learning

A classifier contains two sets and matches when `condition <= msgs`. Percepts come from the cave; `can_explore` and similar **internal messages** come from memory. A rule can emit an effector directly (`grab`) or a goal (`home`, `explore`, …), which the controller translates into an action. The simulator receives **one effector per step**.

When neither safe exploration nor a shot at a known target is available, the *retreat* rule competes with a rule that tries an uncertain square. We use two risk rules: one before the agent has visited half of the 16 squares, and one once it has visited at least half. This separates early risks from later ones without estimating the probability of death.

The agent chooses using **ε-greedy** selection and stores that choice as an internal message so it does not change course after every turn. If it chooses to take a risk, it still prioritizes any new safe moves it discovers. At the end of the episode, we update only the chosen rule's **average return from the point of the decision** (undiscounted Monte Carlo). The other rules stay fixed.

In [4]:
@dataclass(frozen=True)
class Classifier:
    condition: frozenset
    action: frozenset

    def matches(self, msgs):
        return self.condition <= msgs


def rule(condition, action):
    return Classifier(frozenset(condition.split()), frozenset(action.split()))


PRIORITY = [
    rule("has_gold at_start", "climb"),
    rule("has_gold", "home"),
    rule("return_empty", "home"),
    rule("glitter", "grab"),
    rule("can_explore", "explore"),
    rule("can_hunt", "hunt"),
    rule("risk_mode can_risk", "risk"),
]
RETREAT = rule("", "home return_empty")
RISK_EARLY = rule("can_risk early", "risk risk_mode")
RISK_LATE = rule("can_risk late", "risk risk_mode")


class RuleValues:
    def __init__(self):
        self.count = Counter()
        self.mean = {r: 0.0 for r in (RETREAT, RISK_EARLY, RISK_LATE)}

    def choose(self, choices, rng, epsilon):
        if epsilon and rng.random() < epsilon:
            return rng.choice(choices)
        return max(choices, key=self.mean.get)  # RETREAT wins ties.

    def update(self, chosen, episode_return):
        self.count[chosen] += 1
        self.mean[chosen] += (episode_return - self.mean[chosen]) / self.count[chosen]


def decide(signals, memory, values, rng, epsilon):
    options = proposals(memory)
    msgs = signals | memory.flags | {f"can_{goal}" for goal in options}
    msgs.add("late" if len(memory.seen) >= len(BOARD) // 2 else "early")
    selected = next((r for r in PRIORITY if r.matches(msgs)), None)
    learned_choice = None
    if selected is None:
        choices = [r for r in (RETREAT, RISK_EARLY, RISK_LATE) if r.matches(msgs)]
        selected = values.choose(choices, rng, epsilon)
        if len(choices) > 1:
            learned_choice = selected

    memory.flags.update(selected.action & {"return_empty", "risk_mode"})
    effectors = selected.action & EFFECTORS
    if effectors:
        (command,) = effectors
    else:
        (goal,) = selected.action & options.keys()
        command = options[goal]
    return command, learned_choice

## 5. Running an episode

Each episode is capped at 120 actions to stop loops. Hitting the cap counts as a `timeout`, with no extra reward or penalty. Only episodes that end without a timeout are used for Monte Carlo updates.

In [5]:
def play(layout, values, rng=None, epsilon=0, learn=False, max_steps=120):
    cave, memory = Cave(*layout), Memory()
    signals = cave.perceive()
    memory.observe(signals)
    score, credit, trace = 0, None, []
    outcome = None
    for _ in range(max_steps):
        action, choice = decide(signals, memory, values, rng, epsilon)
        if choice is not None:
            credit = choice, score
        trace.append((memory.pos, tuple(sorted(signals)), action))
        signals, reward, outcome = cave.step(action)
        score += reward
        if outcome is not None:
            break
        memory.observe(signals, action)

    if learn and outcome is not None and credit is not None:
        chosen, score_before_choice = credit
        values.update(chosen, score - score_before_choice)
    return outcome or "timeout", score, trace

## 6. Checks and demo on the lab map

The assertions check percepts, scoring, and inference. The demo prints the position **tracked by the agent**, not the position stored by the simulator.

In [6]:
cave = Cave(*EXAMPLE)
memory = Memory()
signals, _, _ = cave.step("turn_r")
memory.observe(signals, "turn_r")
signals, reward, _ = cave.step("fwd")
memory.observe(signals, "fwd")
assert signals == {"at_start", "bump"} and reward == -1
assert memory.pos == START and memory.heading == 3

cave = Cave((2, 1), (3, 1), ())
signals, reward, _ = cave.step("shoot")
assert "scream" in signals and reward == -11
signals, _, _ = cave.step("fwd")
assert signals == {"glitter"}
cave.step("grab")
cave.step("turn_l")
cave.step("turn_l")
cave.step("fwd")
assert cave.step("climb")[1:] == (999, "gold")

cave = Cave((3, 1), (4, 4), ((2, 1),))
assert cave.step("fwd")[1:] == (-1001, "death")
assert Cave(*EXAMPLE).step("climb")[1:] == (-1, "retreat")

memory = Memory()
memory.seen = {START: set(), (2, 1): {"breeze"}, (1, 2): {"stench"}}
pits, wumpus, confirmed_pits = memory.hazards()
assert (2, 2) in BOARD - pits - wumpus
assert (3, 1) in confirmed_pits and wumpus == {(1, 3)}
assert rule("", "home").matches(set())
assert rule("glitter", "grab").matches({"glitter", "breeze"})

outcome, score, trace = play(EXAMPLE, RuleValues())
assert outcome == "gold"
for i, (cell, signals, action) in enumerate(trace, 1):
    print(f"{i:2d}  {cell}  {', '.join(signals) or '—':24s} → {action}")
print(f"\nOutcome: {outcome} | score: {score} | actions: {len(trace)}")

 1  (1, 1)  at_start                 → fwd
 2  (2, 1)  breeze                   → turn_l
 3  (2, 1)  breeze                   → turn_l
 4  (2, 1)  breeze                   → fwd
 5  (1, 1)  at_start                 → turn_r
 6  (1, 1)  at_start                 → fwd
 7  (1, 2)  stench                   → turn_r
 8  (1, 2)  stench                   → fwd
 9  (2, 2)  —                        → fwd
10  (3, 2)  breeze                   → turn_l
11  (3, 2)  breeze                   → turn_l
12  (3, 2)  breeze                   → fwd
13  (2, 2)  —                        → turn_r
14  (2, 2)  —                        → fwd
15  (2, 3)  breeze, glitter, stench  → grab
16  (2, 3)  breeze, stench           → turn_l
17  (2, 3)  breeze, stench           → turn_l
18  (2, 3)  breeze, stench           → fwd
19  (2, 2)  —                        → turn_r
20  (2, 2)  —                        → fwd
21  (1, 2)  stench                   → turn_l
22  (1, 2)  stench                   → fwd
23  (1, 1)  at_start

## 7. Training and testing on unseen caves

We compare two agents on the **same** test caves: a cautious baseline that retreats in a conflict, and an agent that chooses using learned values. During testing, we disable ε-greedy exploration and stop updating the values. Read mean score and outcome counts together: deaths carry a large penalty, and some random caves have no feasible path to the gold.

In [7]:
values = RuleValues()
training_rng, action_rng, test_rng = Random(5), Random(19), Random(50)
train_layouts = [random_layout(training_rng) for _ in range(1600)]

for layout in train_layouts:
    play(layout, values, action_rng, epsilon=0.4, learn=True)

assert all(values.count[r] > 0 for r in values.mean)
for label, classifier in [
    ("retreat", RETREAT),
    ("early risk", RISK_EARLY),
    ("late risk", RISK_LATE),
]:
    print(
        f"{label:16s}: estimated return {values.mean[classifier]:7.1f}"
        f"  ({values.count[classifier]} updates)"
    )

retreat         : estimated return    -1.8  (826 updates)
early risk      : estimated return  -520.3  (199 updates)
late risk       : estimated return   421.7  (71 updates)


In [8]:
used = set(train_layouts)
test_layouts = []
while len(test_layouts) < 500:
    layout = random_layout(test_rng)
    if layout not in used:
        test_layouts.append(layout)
        used.add(layout)
print()
print(
    f"{'Policy':12s} {'Mean':>8s} {'Gold':>6s} {'Deaths':>6s} {'Retreats':>9s} {'Timeouts':>8s}"
)
for label, policy in [("Cautious", RuleValues()), ("Learned", values)]:
    results = [play(layout, policy) for layout in test_layouts]
    counts = Counter(outcome for outcome, _, _ in results)
    average = sum(score for _, score, _ in results) / len(results)
    print(
        f"{label:12s} {average:8.1f} {counts['gold']:6d} {counts['death']:6d}"
        f" {counts['retreat']:9d} {counts['timeout']:8d}"
    )


Policy           Mean   Gold Deaths  Retreats Timeouts
Cautious        296.2    154      0       346        0
Learned         306.0    168      9       323        0


In this run, the agent learns to retreat after seeing only a few squares and to take risks later. It brings back gold more often and earns a higher mean score, but also dies more often.

This comparison measures the effect of **this** learned decision; it does not show that the policy is optimal. Even within each phase, some uncertain squares are more dangerous than others. The rules learn an average return, not the probability of each hazard.